In [2]:
import sys, json
sys.path.append("..")
import numpy as np
import pandas as pd
from src.ensembles import soft_vote, stack_oof, stacking_weights, fold_table, summarize

y = pd.read_parquet("../data/processed/y_train.parquet")["_MICHD"].values
tmp = pd.read_parquet("../data/processed/oof_probabilities.parquet")
oof = {c: tmp[c].values for c in tmp.columns}
print({k: v.shape for k, v in oof.items()})       # three arrays of 361971

{'logreg': (361971,), 'xgb': (361971,), 'rf': (361971,)}


In [3]:
base = ["logreg", "xgb", "rf"]
preds = dict(oof)
preds["voting_lr_xgb_rf"] = soft_vote(oof, base)
preds["voting_xgb_rf"] = soft_vote(oof, ["xgb", "rf"])
preds["stacking_lr_xgb_rf"] = stack_oof(oof, base, y)

folds = fold_table(preds, y)
summ = summarize(folds, reference="xgb")
summ.round(4)

,model,roc_auc,roc_auc_diff,roc_auc_spread,pr_auc,pr_auc_diff,pr_auc_spread,brier,brier_diff,brier_spread
0,voting_xgb_rf,0.8304,0.0001,0.0002,0.3412,0.0005,0.0005,0.0718,0.0000,0.0000
1,stacking_lr_xgb_rf,0.8306,0.0002,0.0001,0.3412,0.0005,0.0003,0.0718,-0.0000,0.0000
2,xgb,0.8303,0.0000,0.0000,0.3407,0.0000,0.0000,0.0718,0.0000,0.0000
3,voting_lr_xgb_rf,0.8304,0.0000,0.0002,0.3405,-0.0002,0.0004,0.0718,0.0000,0.0000
4,rf,0.8287,-0.0016,0.0004,0.3384,-0.0022,0.0010,0.0721,0.0003,0.0000
5,logreg,0.8269,-0.0034,0.0003,0.3356,-0.0050,0.0011,0.0722,0.0004,0.0001


In [4]:
w, b = stacking_weights(oof, base, y)
print(w.round(3).to_string(), "\nintercept:", round(b, 3))

summ.round(5).to_csv("../reports/ensemble_comparison.csv", index=False)

logreg   -0.010
xgb       0.696
rf        0.364 
intercept: 0.089


In [5]:
from src.thresholds import threshold_for_recall, operating_point

p = oof["xgb"]
t90 = float(threshold_for_recall(y, p, 0.90))
t80 = float(threshold_for_recall(y, p, 0.80))
print("t80 =", round(t80, 4), "  t90 =", round(t90, 4))
print({k: round(v, 3) for k, v in operating_point(y, p, t80).items()})

json.dump({"model": "xgb",
           "flag_threshold": t80,
           "moderate_threshold": t90,
           "rule": "highest threshold with out-of-fold recall >= 0.80 (training data only)"},
          open("../reports/final_decision.json", "w"), indent=2)

t80 = 0.0925   t90 = 0.0513
{'threshold': 0.093, 'recall': 0.8, 'specificity': 0.71, 'precision': 0.221, 'flagged_pct': np.float64(33.806), 'flagged_per_case_found': 4.516}


In [6]:
band = np.where(p < t90, "Low", np.where(p < t80, "Moderate", "High"))
tab = (pd.DataFrame({"band": band, "case": y}).groupby("band")["case"]
         .agg(people="size", cases="sum", observed_rate="mean"))
tab["share_of_people_%"] = 100 * tab["people"] / len(y)
tab["share_of_cases_%"] = 100 * tab["cases"] / y.sum()
tab.loc[["Low", "Moderate", "High"]].round(3)

,people,cases,observed_rate,share_of_people_%,share_of_cases_%
band,,,,,
Low,191886,3387,0.018,53.011,10.0
Moderate,47717,3387,0.071,13.183,10.0
High,122368,27096,0.221,33.806,80.0


## Ensemble findings and final model

Voting and stacking ensembles were built from out-of-fold predictions (soft voting = mean probability; stacking = logistic-regression meta-learner on logit probabilities, scored with 5-fold CV) and compared with the single models on identical row subsets (paired differences vs XGBoost).

**Ensembles add nothing.** Stacking (LR + XGB + RF) improved ROC-AUC by 0.0002 and PR-AUC by 0.0005 over XGBoost, voting (XGB + RF) by 0.0001 and 0.0005; Brier scores were identical (0.0718). These gains are far below the 0.002 ROC-AUC practical threshold. The stacker's weights (XGBoost 0.70, Random Forest 0.36, logistic regression −0.01) show it effectively averages the two tree models and discards the linear one.

**Single-model ranking.** XGBoost > Random Forest (−0.0016 ROC-AUC, −0.0022 PR-AUC) > logistic regression (−0.0034, −0.0050); each gap is several times its subset-to-subset spread, although small in absolute terms.

**Decision.** The final model is the tuned, unweighted XGBoost on the 14-feature set: best single model, ensembles offer no meaningful benefit, and it is simple to explain (SHAP) and deploy. Logistic regression is retained as an interpretable benchmark.

**Operating point.** The flag threshold is the highest threshold with out-of-fold recall ≥ 80%, fixed on training data before any test-set evaluation.